# HORIZON - training notebook

Trains the world model and writes the artifacts the backend (`horizon-api/`) loads:

| file | required | consumed by |
| --- | --- | --- |
| `states.parquet` | yes | history slicing, host list, surprise |
| `scaler.json` | yes | feature transforms |
| `model.pt` | yes | the model |
| `metrics.json` | no | `GET /api/metrics` |
| `scenarios.json` | no | demo-host ground truth |
| `model_heldout_<class>.pt` | no | held-out-class surprise overlay |

Everything is written to **`artifacts/`** in the working dir. Download that folder and
drop it into `horizon-api/artifacts/` (see the last cell).

---

## Run in Kaggle

1. New Notebook -> **Add Input** -> search **`pshikk/cicids2017-untampered`** -> Add.
   (Must be the variant WITH `Source IP` / `Timestamp`. `chethuhn/network-intrusion-dataset`,
   `kk0105/cicids2017` and most others are the MachineLearningCVE variant that strips
   those - Gate 0 will fail. Backup slug: `rayenbal/cicids2017`.)
2. Settings -> Internet **ON** (needed for `git clone`), Accelerator: **GPU T4** (optional).
3. Run all. Artifacts appear in `/kaggle/working/artifacts/`; download from the Output tab.

## Run in Colab

1. Runtime -> Change runtime type -> **T4 GPU** (optional).
2. Run the first cells. When prompted, upload your `kaggle.json`
   (kaggle.com -> Account -> Create New API Token) so the dataset can download.
3. Artifacts appear in `/content/artifacts/`; the last cell zips them for download.

## Prerequisite

`horizon-api/` must be pushed to the GitHub repo below - the notebook clones it so
training and serving share one model definition. If you have not pushed it yet, do
that first.

In [ ]:
# ============================ CONFIG ============================
REPO_URL            = "https://github.com/haragam22/HORIZON.git"

# Must be the GeneratedLabelledFlows / TrafficLabelling variant - the 8 weekday
# *_ISCX.csv files WITH Flow ID / Source IP / Destination IP / Timestamp.
# NOT the MachineLearningCVE variant (chethuhn/..., kk0105/..., most others) which
# strips those columns -> Gate 0 fails.
KAGGLE_DATASET      = "pshikk/cicids2017-untampered"   # backup: "rayenbal/cicids2017"

HELDOUT_CAPTURE     = "ids2017-wednesday"   # domain-shift day; held out. NOT friday - friday has all the Bot/PortScan/DDoS.
WINDOW_SECONDS      = 60
MIN_WINDOWS         = 40                 # drop hosts with fewer real windows
WARMUP_WINDOWS      = 5                  # skip first N windows/host when fitting the scaler (new_peer_rate bias)
LABEL_MIN_MALICIOUS = 1                  # malicious flows in a window before it counts as an attack window

EPOCHS              = 40                 # 15 for a fast smoke run
BATCH_SIZE          = 256
LR                  = 1e-3
LAMBDA_BCE          = 1.0
N_ROLLOUT_SAMPLES   = 50

QUICK               = False             # True -> sample 15% of flows for a fast smoke run
RUN_HELDOUT_CLASS   = False             # True -> also retrain with one attack class removed (slow: +1 full train)
HELDOUT_CLASS       = "Bot"              # richest class (589 windows); the friday botnet demo host uses it
SEED                = 0

In [ ]:
import os, sys, pathlib, json, time
import numpy as np, pandas as pd
import torch

torch.manual_seed(SEED); np.random.seed(SEED)

IN_KAGGLE = pathlib.Path("/kaggle").exists()
IN_COLAB  = "google.colab" in sys.modules or pathlib.Path("/content").exists()
WORK = pathlib.Path("/kaggle/working" if IN_KAGGLE else "/content" if IN_COLAB else ".").resolve()
OUT  = WORK / "artifacts"; OUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"env: {'kaggle' if IN_KAGGLE else 'colab' if IN_COLAB else 'local'} | device: {DEVICE} | out: {OUT}")

In [ ]:
# ---- dataset location ----
_slug = KAGGLE_DATASET.split("/")[-1]
if IN_KAGGLE:
    # Kaggle mounts an added dataset at /kaggle/input/<slug>
    cand = [pathlib.Path(f"/kaggle/input/{_slug}"), *pathlib.Path("/kaggle/input").glob("*")]
    DATA = next((p for p in cand if p.exists() and any(p.glob("**/*.csv"))), cand[0])
elif IN_COLAB:
    kj = pathlib.Path("/root/.kaggle/kaggle.json")
    if not kj.exists():
        from google.colab import files
        print("upload kaggle.json:")
        up = files.upload()
        kj.parent.mkdir(parents=True, exist_ok=True)
        kj.write_bytes(list(up.values())[0]); kj.chmod(0o600)
    os.system("pip -q install kaggle")
    os.system(f"kaggle datasets download -d {KAGGLE_DATASET} -p /content/data --unzip")
    DATA = pathlib.Path("/content/data")
else:
    DATA = pathlib.Path("./data")   # local: put the CSVs here

# the 8 canonical weekday files; ignore any pre-merged / sampled extras
csvs = sorted(
    p for p in DATA.glob("**/*.csv")
    if "ISCX" in p.name or any(d in p.name.lower() for d in
       ("monday", "tuesday", "wednesday", "thursday", "friday"))
)
if not csvs:
    csvs = sorted(DATA.glob("**/*.csv"))
assert csvs, f"no CSVs under {DATA} - on Kaggle, Add Input '{KAGGLE_DATASET}'"
# de-dupe by basename (some datasets ship the files twice: zipped + extracted)
_seen, uniq = set(), []
for p in csvs:
    if p.name not in _seen:
        _seen.add(p.name); uniq.append(p)
csvs = uniq
print(len(csvs), "files:")
for c in csvs: print("  ", c.name)

In [ ]:
# ---- clone the repo, import the shared model definition ----
if not pathlib.Path("HORIZON").exists():
    assert os.system(f"git clone -q {REPO_URL} HORIZON") == 0, "git clone failed"
sys.path.insert(0, str(pathlib.Path("HORIZON/horizon-api").resolve()))

try:
    from horizon_api import FEATURE_KEYS, HISTORY, HORIZON, N_INPUT, N_FEATURES
    from horizon_api.features import FeatureScaler
    from horizon_api.model import HorizonModel, ModelConfig
    from horizon_api.model import save as save_model
    from horizon_api.rollout import rollout, make_intervention
except ModuleNotFoundError as e:
    raise SystemExit("horizon-api/ not found in the cloned repo - push it to GitHub first.") from e

FEAT = list(FEATURE_KEYS)
print("model contract OK. features:", FEAT)
print("HISTORY", HISTORY, "HORIZON", HORIZON, "N_INPUT", N_INPUT)

## Gate 0 - column verification

The state design needs source/dest IP, dest port, and a timestamp. Some cleaned CIC dumps drop them.

In [ ]:
probe = pd.read_csv(csvs[0], nrows=300, low_memory=False, encoding="latin-1")
probe.columns = probe.columns.str.strip()
NEED = ["Source IP", "Destination IP", "Destination Port", "Timestamp", "Label"]
missing = [c for c in NEED if c not in probe.columns]
print("have:", [c for c in NEED if c in probe.columns])
if missing:
    raise SystemExit(f"Gate 0 FAIL: missing {missing} -> branch B/C, see technical.md section 0")
print("Gate 0 PASS - branch A, proceed")

In [ ]:
import ipaddress
from collections import defaultdict

def capture_of(name: str) -> str:
    n = name.lower()
    for day in ("monday", "tuesday", "wednesday", "thursday", "friday"):
        if day in n:
            return f"ids2017-{day}"
    return "ids2017-unknown"

def is_internal(ip: str) -> bool:
    try:
        return ipaddress.ip_address(ip).is_private
    except ValueError:
        return False

cap_files = defaultdict(list)
for c in csvs:
    cap_files[capture_of(c.name)].append(c)
print({k: [p.name for p in v] for k, v in cap_files.items()})

## Host-window aggregation

Raw flows -> one 10-feature row per internal host per 60s window, per `technical.md` 1.2. Same-weekday files are concatenated before windowing so window indices are continuous.

In [ ]:
USE = ["Source IP", "Destination IP", "Destination Port", "Timestamp", "Flow Duration",
       "Total Length of Fwd Packets", "Total Length of Bwd Packets", "RST Flag Count", "Label"]

def _load_capture(paths):
    frames = []
    for p in paths:
        for chunk in pd.read_csv(p, chunksize=250_000, low_memory=False, encoding="latin-1"):
            chunk.columns = chunk.columns.str.strip()
            if QUICK:
                chunk = chunk.sample(frac=0.15, random_state=SEED)
            chunk = chunk[[c for c in USE if c in chunk.columns]].copy()
            chunk["ts"] = pd.to_datetime(chunk["Timestamp"], dayfirst=True, errors="coerce")
            chunk = chunk.dropna(subset=["ts", "Source IP", "Destination IP"])
            chunk = chunk[chunk["Source IP"].map(is_internal)]
            if not chunk.empty:
                frames.append(chunk)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# demo (capture, host, scenario, true_class). Friday has two: the botnet victim
# and the scanner 172.16.0.1 - the scanner is the one with real lead time.
DEMO_HOSTS = [
    ("ids2017-thursday", "192.168.10.8",  "infiltration", "Infiltration"),
    ("ids2017-friday",   "192.168.10.15", "botnet",       "Bot"),
    ("ids2017-friday",   "172.16.0.1",    "portscan",     "PortScan"),
    ("ids2017-monday",   "192.168.10.9",  "benign",       "benign"),
]
DEMO_HOST_OF = {cap: h for cap, h, _, _ in DEMO_HOSTS}   # first host per capture, for the flow sampler

def aggregate_capture(cap, paths):
    df = _load_capture(paths)
    if df.empty:
        return pd.DataFrame(), None, None, None
    df["dur_s"] = pd.to_numeric(df["Flow Duration"], errors="coerce").fillna(0) / 1e6
    df["b_out"] = pd.to_numeric(df["Total Length of Fwd Packets"], errors="coerce").fillna(0)
    df["b_in"]  = pd.to_numeric(df["Total Length of Bwd Packets"], errors="coerce").fillna(0)
    df["rst"]   = pd.to_numeric(df.get("RST Flag Count", 0), errors="coerce").fillna(0)
    df["ext"]   = ~df["Destination IP"].map(is_internal)
    df["fail"]  = (df["b_in"] == 0) | (df["rst"] > 0)
    df["mal"]   = df["Label"].astype(str).str.upper().str.strip() != "BENIGN"

    t0 = df["ts"].min()
    df["w"] = ((df["ts"] - t0).dt.total_seconds() // WINDOW_SECONDS).astype(int)

    g = df.groupby(["Source IP", "w"], sort=True)
    agg = g.agg(
        n_flows=("ts", "size"),
        n_distinct_dst_ip=("Destination IP", "nunique"),
        n_distinct_dst_port=("Destination Port", "nunique"),
        fail_ratio=("fail", "mean"),
        bytes_out=("b_out", "sum"),
        bytes_in=("b_in", "sum"),
        mean_duration=("dur_s", "mean"),
        external_ratio=("ext", "mean"),
        n_mal=("mal", "sum"),
    ).reset_index().rename(columns={"Source IP": "host"})
    agg["io_ratio"] = agg["bytes_out"] / (agg["bytes_in"] + 1.0)

    # new_peer_rate: running per-host set of destination IPs, in time order
    dsts = (g["Destination IP"].agg(set).reset_index()
            .rename(columns={"Source IP": "host", "Destination IP": "dsts"})
            .sort_values(["host", "w"]))
    seen, npr = {}, {}
    for host, w, s in zip(dsts.host, dsts.w, dsts.dsts):
        known = seen.setdefault(host, set())
        npr[(host, w)] = len(s - known) / max(1, len(s))
        known |= s
    agg["new_peer_rate"] = [npr[(h, w)] for h, w in zip(agg.host, agg.w)]

    # window label = most common malicious class if enough malicious flows, else benign
    mal_rows = df[df["mal"]]
    if len(mal_rows):
        lab = (mal_rows.groupby(["Source IP", "w"])["Label"]
               .agg(lambda x: x.value_counts().index[0]).reset_index()
               .rename(columns={"Source IP": "host", "Label": "label"}))
        agg = agg.merge(lab, on=["host", "w"], how="left")
    else:
        agg["label"] = np.nan
    agg["label"] = agg["label"].where(agg["n_mal"] >= LABEL_MIN_MALICIOUS, "benign").fillna("benign")
    agg["label"] = agg["label"].astype(str).str.strip()
    agg["capture"] = cap

    # --- macro topology: internal host -> internal host adjacency ---
    di = df[~df["ext"]].copy()
    di["dst"] = di["Destination IP"]
    adj = (di.groupby(["Source IP", "dst"]).size().reset_index(name="flows")
           .rename(columns={"Source IP": "src"}))
    ext_out = df[df["ext"]].groupby("Source IP").size().reset_index(name="flows") \
        .rename(columns={"Source IP": "src"})
    ext_out["dst"] = "ext:internet"
    net_edges = pd.concat([adj, ext_out[["src", "dst", "flows"]]], ignore_index=True)

    # --- micro: flow sample for every demo host in this capture ---
    flows_json = {}
    for dh in [h for c, h, _, _ in DEMO_HOSTS if c == cap]:
        if not (df["Source IP"] == dh).any():
            continue
        fs = df[df["Source IP"] == dh].copy()
        weights = np.where(fs["mal"].values, 5.0, 1.0)  # bias toward malicious windows
        take = min(400, len(fs))
        idx = np.random.default_rng(SEED).choice(len(fs), size=take, replace=False,
                                                 p=weights / weights.sum())
        fs = fs.iloc[np.sort(idx)]
        flows_json[dh] = {
            "schema_version": "v4.0", "capture": cap, "host": dh,
            "flows": [{
                "window_idx": int(w),
                "ts": (t0 + pd.Timedelta(seconds=int(w) * WINDOW_SECONDS)).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "dst_ip": str(d), "dst_port": int(pd.to_numeric(p, errors="coerce") or 0),
                "bytes_out": int(bo), "bytes_in": int(bi),
                "label": str(lb).strip(), "internal": not bool(e),
            } for w, d, p, bo, bi, lb, e in zip(
                fs["w"], fs["Destination IP"], fs["Destination Port"],
                fs["b_out"], fs["b_in"], fs["Label"], fs["ext"])],
        }
    return agg, t0, net_edges, flows_json

parts, cap_t0, cap_edges, cap_flows = [], {}, {}, {}
for cap, paths in cap_files.items():
    a, t0, net, flows_json = aggregate_capture(cap, paths)
    cap_t0[cap] = t0
    cap_edges[cap] = net
    cap_flows.update({(cap, h): fj for h, fj in flows_json.items()})
    print(f"{cap:20s} {len(a):7d} host-windows  ({a.host.nunique() if len(a) else 0} hosts)")
    if len(a):
        parts.append(a)
raw_states = pd.concat(parts, ignore_index=True)

In [ ]:
# ---- fill empty windows, drop short hosts, order columns -> states.parquet ----
def finish(raw):
    out = []
    for (cap, host), grp in raw.groupby(["capture", "host"], sort=False):
        grp = grp.set_index("w").sort_index()
        full = grp.reindex(range(int(grp.index.min()), int(grp.index.max()) + 1))
        full["is_empty"] = full["n_flows"].isna()
        full[FEAT] = full[FEAT].fillna(0.0)
        full["label"] = full["label"].fillna("benign")
        if (~full["is_empty"]).sum() < MIN_WINDOWS:
            continue
        full = full.reset_index().rename(columns={"index": "window_idx"})
        full["capture"], full["host"] = cap, host
        t0 = cap_t0[cap]
        full["ts"] = [(t0 + pd.Timedelta(seconds=int(w) * WINDOW_SECONDS)).strftime("%Y-%m-%dT%H:%M:%SZ")
                      for w in full["window_idx"]]
        out.append(full)
    s = pd.concat(out, ignore_index=True)
    return s[["capture", "host", "window_idx", "ts", *FEAT, "is_empty", "label"]] \
        .sort_values(["capture", "host", "window_idx"]).reset_index(drop=True)

states = finish(raw_states)
states.to_parquet(OUT / "states.parquet", index=False)
print(states.shape, "->", OUT / "states.parquet")
print(states.label.value_counts().head(20))

## Scene data: topology + flow sample

`network_<capture>.json` (macro host-comm graph) and `flows_<capture>_<host>.json` (micro per-host flow sample) for the frontend scene. `states.parquet` does not keep peer identity, so these are built here from the raw flows.

In [ ]:
KIND_HINT = {"1": "gateway", "3": "domain-controller", "5": "server", "16": "server", "19": "server"}

def node_kind(host, in_deg, out_deg):
    last = host.rsplit(".", 1)[-1]
    if host.startswith("ext"):
        return "external"
    if last in KIND_HINT:
        return KIND_HINT[last]
    if in_deg >= 8 and in_deg > out_deg * 2:
        return "server"
    return "workstation"

kept = set(states["host"].unique()) | {"ext:internet"}
for cap in cap_files:
    edges_df = cap_edges.get(cap)
    if edges_df is None or not len(edges_df):
        continue
    e = edges_df[edges_df["src"].isin(kept) & edges_df["dst"].isin(kept)]
    e = e[e["flows"] >= 3].sort_values("flows", ascending=False).head(400)
    indeg = e.groupby("dst")["flows"].sum().to_dict()
    outdeg = e.groupby("src")["flows"].sum().to_dict()
    g = states[states["capture"] == cap].groupby("host").agg(
        n_flows=("n_flows", "sum"), n_windows=("window_idx", "count")).to_dict("index")
    hosts_in = sorted(set(e["src"]) | set(e["dst"]))
    dhs = {h for c, h, _, _ in DEMO_HOSTS if c == cap}
    nodes = [{
        "host": h,
        "subnet": "external" if h.startswith("ext") else h.rsplit(".", 1)[0],
        "n_flows": int(g.get(h, {}).get("n_flows", indeg.get(h, 0) + outdeg.get(h, 0))),
        "n_windows": int(g.get(h, {}).get("n_windows", 0)),
        "kind": node_kind(h, indeg.get(h, 0), outdeg.get(h, 0)),
        "is_demo": h in dhs,
        "is_target": node_kind(h, indeg.get(h, 0), outdeg.get(h, 0)) == "domain-controller",
    } for h in hosts_in]
    edges = [{"src": r.src, "dst": r.dst, "flows": int(r.flows),
              "internal": not str(r.dst).startswith("ext")} for r in e.itertuples()]
    net = {"schema_version": "v4.0", "capture": cap, "nodes": nodes, "edges": edges}
    (OUT / f"network_{cap}.json").write_text(json.dumps(net, indent=1))
    print(f"network_{cap}.json: {len(nodes)} nodes, {len(edges)} edges")

for (cap, h), fj in cap_flows.items():
    (OUT / f"flows_{cap}_{h}.json").write_text(json.dumps(fj, indent=1))
    print(f"flows_{cap}_{h}.json: {len(fj['flows'])} flows")

## Transforms

`FeatureScaler` = log1p on the heavy-tailed subset, then standardise. Fit on the **train split only** (`HELDOUT_CAPTURE` excluded), skipping each host's warm-up windows.

In [ ]:
train_cap = states["capture"] != HELDOUT_CAPTURE
pos_in_host = states.groupby(["capture", "host"]).cumcount()
fit_rows = states[train_cap & (pos_in_host >= WARMUP_WINDOWS) & (~states["is_empty"])]
print("scaler fit rows:", len(fit_rows))

scaler = FeatureScaler().fit(fit_rows[FEAT].to_numpy())
scaler.save(OUT / "scaler.json")

Z_ALL = scaler.transform(states[FEAT].to_numpy()).astype(np.float32)
print("z mean:", Z_ALL.mean(0).round(2))
print("z std :", Z_ALL.std(0).round(2))

## Sequences and splits

One training example per window `i >= 20`: history `z[i-20:i]` (+ zero intervention channel), MDN target `z[i]`, readout target = does an attack window fall in `[i, i+20)`. Grouped by `(capture, host)`; `HELDOUT_CAPTURE` is the out-of-domain test set, plus a 15% host slice of the train captures for early stopping.

In [ ]:
host_index = {}
off = 0
for (cap, host), grp in states.groupby(["capture", "host"], sort=False):
    host_index[(cap, host)] = (off, off + len(grp), grp)
    off += len(grp)

rng = np.random.default_rng(SEED)
train_hosts, val_hosts, test_hosts = [], [], []
for key, (_, _, grp) in host_index.items():
    cap = key[0]
    if cap == HELDOUT_CAPTURE:
        test_hosts.append(key)
    elif rng.random() < 0.15:
        val_hosts.append(key)
    else:
        train_hosts.append(key)

def make_xy(keys, drop_class=None):
    X, Ynext, Yatk = [], [], []
    zero_col = np.zeros((HISTORY, 1), dtype=np.float32)
    for key in keys:
        lo, hi, grp = host_index[key]
        z = Z_ALL[lo:hi]
        lab = grp["label"].to_numpy()
        if drop_class is not None and (lab == drop_class).any():
            continue  # held-out-class: remove hosts that ever show the class
        atk = lab != "benign"
        for i in range(HISTORY, len(grp) - 1):
            X.append(np.concatenate([z[i - HISTORY:i], zero_col], axis=1))
            Ynext.append(z[i])
            Yatk.append(float(atk[i:i + HORIZON].any()))
    return (np.asarray(X, np.float32), np.asarray(Ynext, np.float32), np.asarray(Yatk, np.float32))

Xtr, Ntr, Atr = make_xy(train_hosts)
Xva, Nva, Ava = make_xy(val_hosts)
print("train", Xtr.shape, "| val", Xva.shape, "| attack rate tr/va:", Atr.mean().round(3), Ava.mean().round(3))

## MDN sanity check on toy data

Before real training: can the MDN head separate a deliberately bimodal target? If not it will not work on real data (`implementation.md` Week 1).

In [ ]:
_toy_cfg = ModelConfig()
_toy = HorizonModel(_toy_cfg).to(DEVICE)
_opt = torch.optim.Adam(_toy.parameters(), lr=1e-3)
# next value is +2 with p=0.6, -2 with p=0.4, regardless of history
for step in range(300):
    x = torch.randn(256, HISTORY, N_INPUT, device=DEVICE)
    sign = torch.where(torch.rand(256, device=DEVICE) < 0.6, 2.0, -2.0)
    tgt = (sign[:, None] + 0.1 * torch.randn(256, N_FEATURES, device=DEVICE))
    st, *_ = _toy.encode(x)
    loss = _toy.mdn.nll(st, tgt)
    _opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(_toy.parameters(), 1.0)
    _opt.step()
with torch.no_grad():
    st, *_ = _toy.encode(torch.randn(2000, HISTORY, N_INPUT, device=DEVICE))
    samp = _toy.mdn.sample(st)[:, 0].cpu().numpy()
frac_pos = (samp > 0).mean()
print(f"toy: sampled P(+mode) = {frac_pos:.2f}  (target 0.60)  final nll {loss.item():.3f}")
assert 0.45 < frac_pos < 0.75, "MDN did not separate the bimodal toy target - investigate before real training"
print("MDN sanity OK")

## Train

Joint loss `mdn_nll + LAMBDA_BCE * bce`, both logged separately. Scheduled sampling ramps a short model-fed unroll from 0 to 0.5 over training (`technical.md` 2.5). Checkpoint every epoch to `artifacts/`.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def loader(X, N, A, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(N), torch.from_numpy(A))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, drop_last=shuffle)

def train_model(Xtr, Ntr, Atr, Xva, Nva, Ava, tag="model", epochs=EPOCHS):
    torch.manual_seed(SEED)
    model = HorizonModel(ModelConfig()).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    # cap pos_weight: raw n_neg/n_pos (~50) saturates the readout into a step function
    pw = min(8.0, (1 - Atr.mean()) / max(Atr.mean(), 1e-3))
    bce = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pw], device=DEVICE))
    tl, vl = loader(Xtr, Ntr, Atr, True), loader(Xva, Nva, Ava, False)

    best, best_state, bad = 1e9, None, 0
    hist = []
    for ep in range(epochs):
        p_ss = 0.5 * ep / max(1, epochs - 1)
        model.train(); m_sum = b_sum = n = 0
        for xb, nb, ab in tl:
            xb, nb, ab = xb.to(DEVICE), nb.to(DEVICE), ab.to(DEVICE)
            state, ctx, _, hidden = model.encode(xb)
            l_mdn = model.mdn.nll(state, nb)
            l_bce = bce(model.readout(state), ab)
            # scheduled-sampling: one extra step fed by the model's own sample
            if p_ss > 0 and torch.rand(1).item() < p_ss:
                s = model.mdn.sample(state).detach()
                x_next = torch.cat([s, torch.zeros(s.size(0), 1, device=DEVICE)], -1)
                state2, _ = model.step(x_next, hidden, ctx)
                l_bce = l_bce + bce(model.readout(state2), ab)
            loss = l_mdn + LAMBDA_BCE * l_bce
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            m_sum += l_mdn.item(); b_sum += l_bce.item(); n += 1

        model.eval(); vm = vb = vn = 0
        with torch.no_grad():
            for xb, nb, ab in vl:
                xb, nb, ab = xb.to(DEVICE), nb.to(DEVICE), ab.to(DEVICE)
                st, *_ = model.encode(xb)
                vm += model.mdn.nll(st, nb).item()
                vb += bce(model.readout(st), ab).item(); vn += 1
        v = vm / vn + vb / vn
        sched.step(v)
        hist.append({"epoch": ep, "mdn": m_sum / n, "bce": b_sum / n, "val": v, "p_ss": round(p_ss, 3)})
        print(f"ep {ep:2d}  mdn {m_sum/n:7.3f}  bce {b_sum/n:6.3f}  val {v:7.3f}  p_ss {p_ss:.2f}")
        save_model(model, OUT / f"{tag}.pt")
        if v < best - 1e-3:
            best, best_state, bad = v, {k: t.cpu().clone() for k, t in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= 7:
                print("early stop"); break
    if best_state:
        model.load_state_dict(best_state)
        save_model(model, OUT / f"{tag}.pt")
    return model, hist

model, hist = train_model(Xtr, Ntr, Atr, Xva, Nva, Ava)

# ---- Platt calibration: fit sigmoid(a*logit + b) on the val split ----
@torch.no_grad()
def val_logits(m, X):
    out = []
    for i in range(0, len(X), 1024):
        st, *_ = m.encode(torch.from_numpy(X[i:i + 1024]).to(DEVICE))
        out.append(m.readout(st).cpu().numpy())
    return np.concatenate(out)

from sklearn.linear_model import LogisticRegression
_lg = val_logits(model, Xva).reshape(-1, 1)
_pl = LogisticRegression(C=1e6).fit(_lg, Ava.astype(int))
PLATT = {"a": float(_pl.coef_[0, 0]), "b": float(_pl.intercept_[0])}
json.dump(PLATT, open(OUT / "platt.json", "w"), indent=1)
_praw = 1 / (1 + np.exp(-_lg[:, 0]))
_pcal = 1 / (1 + np.exp(-(PLATT["a"] * _lg[:, 0] + PLATT["b"])))
print(f"platt.json: a={PLATT['a']:.3f} b={PLATT['b']:.3f}  "
      f"mean p raw {_praw.mean():.3f} -> cal {_pcal.mean():.3f}  (val attack rate {Ava.mean():.3f})")

## Persistence gate

Hard gate (`technical.md` 3.1): the model must beat `s_hat_{t+1} = s_t` on per-feature MSE. If not, it collapsed to the mean.

In [ ]:
@torch.no_grad()
def per_feature_mse(model, X, N):
    model.eval()
    preds, base = [], []
    for i in range(0, len(X), 1024):
        xb = torch.from_numpy(X[i:i + 1024]).to(DEVICE)
        st, *_ = model.encode(xb)
        # mean of the MDN over components as the point prediction
        logits, means, _ = model.mdn(st)
        w = torch.softmax(logits, -1).unsqueeze(-1)
        preds.append((w * means).sum(1).cpu().numpy())
        base.append(xb[:, -1, :N_FEATURES].cpu().numpy())
    pred = np.concatenate(preds); last = np.concatenate(base)
    mdl = ((pred - N) ** 2).mean(0)
    per = ((last - N) ** 2).mean(0)
    return mdl, per

mdl_mse, per_mse = per_feature_mse(model, Xva, Nva)
beat = mdl_mse < per_mse
print(f"{'feature':22s} {'model':>8s} {'persist':>8s}  beat")
for k, m, p, b in zip(FEAT, mdl_mse, per_mse, beat):
    print(f"{k:22s} {m:8.4f} {p:8.4f}  {'yes' if b else 'NO'}")
print(f"\nGATE: model beats persistence on {beat.sum()}/{len(FEAT)} features")
if beat.sum() < len(FEAT) * 0.6:
    print("WARNING: weak - train longer (EPOCHS=40), check scaler leakage, or QUICK=False")

## Rollout check + metrics.json

Run the real 50-sample rollout on a few hosts, sanity-check the curves, write `metrics.json` (persistence table + attack-rate calibration). The full lead-time-vs-FPR curve is P2's Week-3 harness; this is the subset the panel needs.

In [ ]:
demo_hosts = [(c, h) for c, h, _, _ in DEMO_HOSTS]

model.eval()
for cap, host in demo_hosts:
    key = (cap, host)
    if key not in host_index:
        print(f"{cap}/{host}: not in states (skipped)"); continue
    lo, hi, grp = host_index[key]
    z = Z_ALL[lo:hi]
    if len(z) < HISTORY + 5:
        print(f"{cap}/{host}: too short"); continue
    # scan a window near the host's first attack (or mid-series) so the curve is interesting
    lab = (grp["label"].values != "benign")
    onset = int(np.argmax(lab)) if lab.any() else len(z) // 2
    for t in [max(HISTORY, onset - 10), max(HISTORY, onset - 3), min(len(z) - 1, onset + 5)]:
        res = rollout(model, z[t - HISTORY:t], n_samples=N_ROLLOUT_SAMPLES, seed=0,
                      platt=(PLATT["a"], PLATT["b"]))
        print(f"{cap}/{host} t={t:4d} (onset {onset}): p_mean {res.p_mean[0]:.2f}..{res.p_mean[-1]:.2f}"
              f"  p_frac peak {res.p_frac.max():.2f}  div {res.divergence:.3f}")

# calibration reliability on val, AFTER Platt
_lg2 = _lg[:, 0]
pv = 1 / (1 + np.exp(-(PLATT["a"] * _lg2 + PLATT["b"])))
bins = np.linspace(0, 1, 6)
reliability = []
for a, b in zip(bins[:-1], bins[1:]):
    m = (pv >= a) & (pv < b)
    if m.sum():
        reliability.append({"p_pred": round((a + b) / 2, 2), "p_obs": round(float(Ava[m].mean()), 3),
                            "n": int(m.sum())})

metrics = {
    "schema_version": "v4.0",
    "status": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "lead_time_vs_fpr": [],  # filled by P2's harness
    "reconstruction": {
        "persistence_beaten": bool(beat.sum() >= len(FEAT) * 0.6),
        "per_feature_mse": {
            "HORIZON": {k: round(float(v), 4) for k, v in zip(FEAT, mdl_mse)},
            "persistence": {k: round(float(v), 4) for k, v in zip(FEAT, per_mse)},
        },
    },
    "rollout_error_growth": [],
    "generalisation": {"held_out_capture": {}, "held_out_class": {}},
    "standard": {"per_class_f1": {}},
    "calibration": {"platt_slope": round(PLATT["a"], 4), "platt": PLATT, "reliability": reliability},
    "divergence_auc": None,
    "training_history": hist,
}
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=1))
print("metrics.json written (base)")

## Evaluation harness

Fills the empty `metrics.json` fields with real numbers: lead-time-vs-false-alarm curve against two baselines, per-class F1 and lead time, held-out-weekday drop, rollout error growth, surprise and divergence AUC. `technical.md` §6.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

CLASSES = sorted(c for c in states["label"].unique() if c != "benign")

# ---- per-window scores over full host sequences (eval = val + held-out weekday) ----
@torch.no_grad()
def host_scores(keys, roll_stride=4):
    # returns list of dicts: capture, host, widx[], label[], p_direct[], p_roll[], surprise[], divergence[]
    out = []
    for key in keys:
        lo, hi, grp = host_index[key]
        z = Z_ALL[lo:hi]
        n = len(grp)
        if n < HISTORY + 2:
            continue
        wid = grp["window_idx"].to_numpy()
        lab = grp["label"].to_numpy()
        atk = lab != "benign"
        idxs = list(range(HISTORY, n - 1))
        # direct readout + surprise for every window (cheap, batched)
        xb = np.stack([np.concatenate([z[i - HISTORY:i], np.zeros((HISTORY, 1), np.float32)], 1) for i in idxs])
        st, *_ = model.encode(torch.from_numpy(xb).to(DEVICE))
        p_dir = torch.sigmoid(PLATT["a"] * model.readout(st) + PLATT["b"]).cpu().numpy()
        nll = model.mdn.per_feature_nll(st, torch.from_numpy(z[idxs]).to(DEVICE)).sum(-1)
        surprise = nll.cpu().numpy()
        # rollout on a stride (expensive)
        p_roll = np.full(len(idxs), np.nan)
        diverg = np.full(len(idxs), np.nan)
        for j in range(0, len(idxs), roll_stride):
            i = idxs[j]
            r = rollout(model, z[i - HISTORY:i], n_samples=30, seed=0, platt=(PLATT["a"], PLATT["b"]))
            p_roll[j] = r.p_frac.max()
            diverg[j] = r.divergence
        out.append(dict(capture=key[0], host=key[1], widx=wid[idxs], label=lab[idxs],
                        atk=atk[idxs], p_dir=p_dir, p_roll=p_roll, surprise=surprise, diverg=diverg,
                        onset=(int(wid[idxs][np.argmax(atk[idxs])]) if atk[idxs].any() else None)))
    return out

eval_scores = host_scores(val_hosts + test_hosts)
print(f"eval hosts scored: {len(eval_scores)}")

# ---- logistic regression baseline (flat last-window features) ----
lr = LogisticRegression(max_iter=800, class_weight="balanced").fit(Xtr[:, -1, :N_FEATURES], Atr.astype(int))
def lr_scores(hs):
    for h in hs:
        lo, hi, grp = host_index[(h["capture"], h["host"])]
        z = Z_ALL[lo:hi]
        i0 = HISTORY
        h["p_lr"] = lr.predict_proba(z[i0:i0 + len(h["widx"]), :N_FEATURES])[:, 1]
lr_scores(eval_scores)

# ---- lead-time vs per-host false-alarm rate ----
def curve(score_key):
    rows = []
    for thr in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
        leads, fp_hosts, benign_hosts = [], 0, 0
        for h in eval_scores:
            s = h[score_key]
            fires = np.where(s >= thr)[0]
            fires_w = int(h["widx"][fires[0]]) if len(fires) else None
            if h["onset"] is None:
                benign_hosts += 1
                if fires_w is not None:
                    fp_hosts += 1
            elif fires_w is not None and fires_w <= h["onset"] + HORIZON:
                leads.append(h["onset"] - fires_w)
        rows.append({"fpr": round(fp_hosts / max(benign_hosts, 1), 4),
                     "lead_windows": round(float(np.mean(leads)) if leads else 0.0, 2),
                     "threshold": thr})
    return rows

lead_curve = []
c_h, c_d, c_l = curve("p_roll"), curve("p_dir"), curve("p_lr")
for a, b, d in zip(c_h, c_d, c_l):
    lead_curve.append({"fpr": b["fpr"], "lead_windows": {
        "HORIZON": a["lead_windows"], "direct_classifier": b["lead_windows"], "logreg": d["lead_windows"]}})

# ---- per-class F1 + per-class lead time (at threshold 0.5, direct readout) ----
per_class_f1, per_class_lead = {}, {}
for cls in CLASSES:
    y_true, y_pred, leads = [], [], []
    for h in eval_scores:
        m = (h["label"] == cls)
        y_true += list(m.astype(int)); y_pred += list((h["p_dir"] >= 0.5).astype(int))
        if m.any():
            onset_c = int(h["widx"][np.argmax(m)])
            fires = np.where(h["p_dir"] >= 0.5)[0]
            if len(fires):
                leads.append(onset_c - int(h["widx"][fires[0]]))
    if sum(y_true) >= 3:
        per_class_f1[cls] = round(float(f1_score(y_true, y_pred, zero_division=0)), 3)
        if leads:
            per_class_lead[cls] = round(float(np.mean(leads)), 2)

# ---- held-out weekday drop (F1 window-level) ----
def f1_of(hs, key):
    yt, yp = [], []
    for h in hs:
        yt += list(h["atk"].astype(int)); yp += list((h[key] >= 0.5).astype(int))
    return round(float(f1_score(yt, yp, zero_division=0)), 3) if yt else None
IN  = [h for h in eval_scores if h["capture"] != HELDOUT_CAPTURE]
OUT_ = [h for h in eval_scores if h["capture"] == HELDOUT_CAPTURE]
f1_in, f1_out = f1_of(IN, "p_dir"), f1_of(OUT_, "p_dir")
f1_lr_in = f1_of(IN, "p_lr")

# ---- rollout error growth (teacher-forcing off) ----
growth = []
samp = [k for k in train_hosts if host_index[k][1] - host_index[k][0] > HISTORY + 25][:40]
errs = {1: [], 5: [], 10: [], 20: []}
for key in samp:
    lo, hi, grp = host_index[key]
    z = Z_ALL[lo:hi]
    for i in range(HISTORY, len(grp) - 20, 15):
        r = rollout(model, z[i - HISTORY:i], n_samples=20, seed=0)
        tm = r.trajectory_mean_z()
        for k in errs:
            if i + k - 1 < len(z):
                errs[k].append(float(((tm[k - 1] - z[i + k - 1]) ** 2).mean()))
for k in sorted(errs):
    growth.append({"step": k, "mse": round(float(np.mean(errs[k])) if errs[k] else 0.0, 4), "nll": None})

# ---- surprise + divergence AUC ----
s_all = np.concatenate([h["surprise"] for h in eval_scores])
y_all = np.concatenate([h["atk"].astype(int) for h in eval_scores])
surprise_auc = round(float(roc_auc_score(y_all, s_all)), 3) if y_all.sum() else None
dv = np.concatenate([h["diverg"] for h in eval_scores]); dm = ~np.isnan(dv)
# divergence label: attack anywhere in this host's remaining sequence
dy = np.concatenate([np.full(len(h["diverg"]), int(h["onset"] is not None)) for h in eval_scores])
divergence_auc = round(float(roc_auc_score(dy[dm], dv[dm])), 3) if dm.sum() and dy[dm].sum() else None

metrics["lead_time_vs_fpr"] = lead_curve
metrics["rollout_error_growth"] = growth
metrics["generalisation"]["held_out_capture"] = {
    "macro_f1_in": f1_in, "macro_f1_out": f1_out,
    "lead_time_drop_windows": None}
metrics["standard"]["per_class_f1"] = per_class_f1
metrics["standard"]["per_class_lead_windows"] = per_class_lead
metrics["standard"]["HORIZON"] = {"macro_f1": f1_in}
metrics["standard"]["direct_classifier"] = {"macro_f1": f1_in}  # same head, no rollout
metrics["standard"]["logreg"] = {"macro_f1": f1_lr_in}
metrics["divergence_auc"] = divergence_auc
metrics["generalisation"]["held_out_class"] = {}
metrics["surprise_auc"] = surprise_auc
metrics["status"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=1))

print("lead curve:", lead_curve)
print("per-class F1:", per_class_f1)
print("per-class lead:", per_class_lead)
print(f"held-out weekday macro-F1: in {f1_in}  out {f1_out}")
print("rollout error growth:", growth)
print(f"surprise AUC {surprise_auc}  divergence AUC {divergence_auc}")
print("metrics.json updated with real eval numbers")

## scenarios.json

Ground truth for the demo hosts, read from the real aggregated data. The backend merges this over its built-in table.

In [ ]:
# held_out class per demo host = the class removed in the RUN_HELDOUT_CLASS run.
HELD_HINT = {("ids2017-friday", "192.168.10.15"): HELDOUT_CLASS}

def first_sustained_attack(grp):
    # first window that begins a run of >=2 malicious windows - skips isolated
    # early scan blips so the number reflects the real attack onset
    lab = (grp.sort_values("window_idx")["label"].values != "benign")
    wid = grp.sort_values("window_idx")["window_idx"].values
    for i in range(len(lab) - 1):
        if lab[i] and lab[i + 1]:
            return int(wid[i])
    return int(wid[lab.argmax()]) if lab.any() else None

scenarios = {}
for cap, host, scen, cls in DEMO_HOSTS:
    key = (cap, host)
    if key not in host_index:
        print(f"  {cap}/{host}: NOT in states - check DEMO_HOSTS"); continue
    _, _, grp = host_index[key]
    n = len(grp)
    first = first_sustained_attack(grp)
    if first is not None:
        avail = sorted({max(HISTORY, first - 12), max(HISTORY, first - 4), min(n - 2, first + 15)})
    else:
        avail = sorted({HISTORY + 5, n // 2, max(HISTORY + 5, n - 20)})
    scenarios[f"{cap}/{host}"] = {
        "scenario": scen,
        "true_class": cls,
        "first_attack_window": first,
        "available_t": avail,
        "held_out": HELD_HINT.get(key),
    }
(OUT / "scenarios.json").write_text(json.dumps(scenarios, indent=1))
print(json.dumps(scenarios, indent=1))

## Held-out-class experiment (optional)

Set `RUN_HELDOUT_CLASS = True` in config. Retrains from scratch with every host that ever shows `HELDOUT_CLASS` removed, saves `model_heldout_<class>.pt`. The backend uses it for the surprise overlay - does surprise still flag the unseen class?

In [ ]:
if RUN_HELDOUT_CLASS:
    Xh, Nh, Ah = make_xy(train_hosts, drop_class=HELDOUT_CLASS)
    Xhv, Nhv, Ahv = make_xy(val_hosts, drop_class=HELDOUT_CLASS)
    print(f"held-out '{HELDOUT_CLASS}': train {Xh.shape}")
    ho_model, _ = train_model(Xh, Nh, Ah, Xhv, Nhv, Ahv, tag=f"model_heldout_{HELDOUT_CLASS}")
    print(f"saved model_heldout_{HELDOUT_CLASS}.pt")
else:
    print("skipped (RUN_HELDOUT_CLASS = False)")

## Package the artifacts

Download `artifacts/` and drop its contents into `horizon-api/artifacts/`, then restart uvicorn (see `horizon-api/README.md` step 3).

In [ ]:
import shutil
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:32s} {f.stat().st_size/1024:8.1f} KB")

zip_path = shutil.make_archive(str(WORK / "horizon_artifacts"), "zip", OUT)
print("\nzip:", zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
elif IN_KAGGLE:
    print("Kaggle: download from the Output tab (artifacts/ and horizon_artifacts.zip).")
else:
    print("local: artifacts are in", OUT)